# Island 1 of TSET-As-1, constant-interaction model, T = 0

**Only classes and functions here**

Constant-interaction model (Beenakker Paper) + a ladder of single-particle levels.

In [1]:
import numpy as np
from dataclasses import dataclass

## 1. UNITS (Atomic Units used)

Here we use atomic units so $\hbar = e = m_e = 4\pi\epsilon_0 = 1$ and the unit of energy is the Hartree (27.2114 eV) and the unit of length is the Bohr radius (0.0529 nm).

We define units that will be used throughout

In [2]:
_E, _HA, _HA_EV, _A0 = 1.602176634e-19, 4.3597447222071e-18, 27.211386245988, 5.29177210903e-11

eV  = 1.0 / _HA_EV                 # 1 eV in atomic units
meV = 1e-3 * eV
nm  = 1e-9 / _A0
V   = _E / _HA                     # volt
mV  = 1e-3 * V
F   = _HA / _E**2                  # farad
aF  = 1e-18 * F
_HBAR, _KB = 1.054571817e-34, 1.380649e-23
sec = _HA / _HBAR                  # second (atomic unit of time is hbar/E_h)
Hz  = 1.0 / sec
MHz = 1e6 * Hz
Kelvin = _KB / _HA                 # k_B * (1 K) in Hartree
mK  = 1e-3 * Kelvin
Ohm = _E**2 / _HBAR             
MOhm = 1e6 * Ohm
uS  = 1e-6 / Ohm                   # microsiemens

## 2. PARAMETERS (used so far)

### The source capacitance $C_S$

The thesis gives $E_C$ and $\alpha_g$, but nothing on how the island's total capacitance is shared between the source and the drain. For now I assume a symmetric island and set $c = C_S/C_\Sigma = 0.5$, so the source gets half of the total capacitance.

$c$ only sets the tilt of the diamonds. With $c = 0.5$ the top and bottom corners line up in the middle of each diamond, and the two edge slopes are equal and opposite, $\pm C_g/C_S = \pm 2\alpha_g = \pm 0.16$. It has no effect on the Coulomb peak positions, the addition energy or the size of the diamonds, because those are fixed by $E_C$, $\alpha_g$ and $\Delta E$.

In [3]:
@dataclass(frozen=True)
class Params:
    E_C     : float = 11.2 * meV   # From Thesis Table 8.3 island 1: the charging energy (e^2/C_sigma)
    alpha_g : float = 0.08         # From Thesis Table 8.3 island 1, gate G1: alpha = C_g/C_sigma
    c_S     : float = 0.5          # C_S/C_sigma, assumed symmetric island (source gets half of the total capacitance)
    g       : int   = 2            # Assume spin only (Beenakker); valley degeneracy not included
    dE      : float = 1.14 * meV   # Energy spacing from g/(D2*A), D2 = 10.4 /eV/nm^2 (Constantinou paper), A = 13x13 nm^2, valid only if valleys are split
    K       : int   = 16           # number of levels kept (holds 2K electrons)

## 2b. PLACEHOLDERS FOR LATER STAGES 

In [4]:
# Nothing below is read by Island or blockade_map. Every parameter the project will need has a slot
# here, so later stages fill values in instead of changing the code. None = value still to be chosen
# Comments give the source of each value that is filled in.

@dataclass(frozen=True)
class GeometryParams:            
    island_size   : float = 13 * nm      # 13 x 13 nm
    gap_S_rows    : int   = 9            # source gap in dimer rows (6.93 nm)
    gap_D_rows    : int   = 8            # drain gap in dimer rows (6.16 nm)
    row_pitch     : float = 0.77 * nm    # per dimer row (table footnote)
    strip_width   : float = 18 * nm      # source/drain strip width
    gate1_width   : float = 100 * nm
    gate1_gap     : float = 69 * nm
    gate2_width   : float = None         # gate 2 is screened for island 1: no values given

@dataclass(frozen=True)
class ElectrostaticsParams:         #For the coupling from the defects to the island
    overgrowth     : float = 21.95 * nm  # Assumed equal to TSET-P-1 (thesis p.212-213); As-1 value not in the pages I have
    eps_r_Si       : float = 11.7        # silicon
    delta_thickness: float = None        # thickness of the delta-layer (ask the experimentalists)
    sheet_density  : float = None        # arsenic donors per unit area (ask the experimentalists)
    oxide_thickness: float = None        # ask the experimentalists
    eps_r_oxide    : float = None
    screening_length: float = None       # Thomas-Fermi screening length of the sheet
    alpha_g2       : float = None        # lever arm of gate 2 on island 1 (possibly will not be used)
    mutual_capacitances: object = None   # full matrix C_ij between island, S, D, G1, G2

@dataclass(frozen=True)
class ThermalParams:                
    T         : float = 0.0              # temperature used in the model (T=0 for now)
    T_base    : float = 7 * mK           # thesis, base temperature of the measurement
    T_electron: float = None             # electron temperature (may exceed the base temperature)

@dataclass(frozen=True)
class TunnelParams:                 # tunnelling to source and drain
    R_total : float = 1.56 * MOhm        # thesis Table 8.3, island 1 (1.56 +- 0.18)
    G_total : float = 0.64 * uS          # thesis Table 8.3, island 1 (0.64 +- 0.07)
    gamma_S : float = None               # rate through the source gap
    gamma_D : float = None               # rate through the drain gap

@dataclass(frozen=True)
class HydrogenicParams:            # used to model the structure defects
    E_b          : float = 53.8 * meV    # arsenic binding energy, thesis p.223 Table 8.1
    m_eff        : float = None          # effective mass for the hydrogenic radius (not done yet)
    attempt_rate : float = None          

@dataclass(frozen=True)
class DefectParams:                 # the density of the defects in the device 
    n_bulk      : float = None           # per unit VOLUME in the bulk
    n_interface : float = None           # per unit AREA at the interface
    n_sheet     : float = None           # traps in the doped sheet/electrodes
    E_win       : float = None           # width of the band of trap levels around the chemical potential
    xi          : float = 2.4 * nm       # tunnelling decay length of the trap wavefunction
    tau0        : float = 1e-12 * sec    # attempt time 
    lam_model   : object = None          # how lambda(r) (coupling to the island) will be compute

@dataclass(frozen=True)
class StrayDopantPopulation:        # one population of stray arsenic dopants for now
    name      : str   = ""
    n_volume  : float = None             # density per unit volume near the sheet (15 Oct notes)
    n_area    : float = None             # density per unit area
    diffusion : float = None             # diffusion parameter: spread through the bulk (15 Oct notes)

@dataclass(frozen=True) 
class StrayDopantParams:           
    incorporation_fraction : float = None   
    populations : tuple = (StrayDopantPopulation("gate_coupled"),     # near the RF lead, strongly gate-coupled
                           StrayDopantPopulation("tunnel_coupled"),   #   tunnel-coupled to the SET
                           StrayDopantPopulation("far"))              #   far charge fluctuators

@dataclass(frozen=True)
class SweepParams:                  # Stage 1 demo uses these ranges and the steps are choices not defined yet
    V_G_max : float = 600 * mV           # thesis: gate stepped between +-600 mV
    V_SD_max: float = 20 * mV            # thesis: source-drain swept between +-20 mV
    n_VG    : int   = None
    n_VSD   : int   = None

@dataclass(frozen=True) 
class NoiseParams:                 
    dt             : float = None        # time step of the simulated telegraph signals
    t_total        : float = None        # simulated duration
    n_realisations : int   = None
    f_min          : float = None        # frequency range
    f_max          : float = None
    seed           : int   = None
    gate_voltage_noise: float = None     # noise of the gate electrodes themselves

@dataclass(frozen=True)
class Device:
    island        : Params = Params()
    geometry      : GeometryParams = GeometryParams()
    electrostatics: ElectrostaticsParams = ElectrostaticsParams()
    thermal       : ThermalParams = ThermalParams()
    tunnel        : TunnelParams = TunnelParams()
    hydrogenic    : HydrogenicParams = HydrogenicParams()
    defects       : DefectParams = DefectParams()
    stray_dopants : StrayDopantParams = StrayDopantParams()
    sweep         : SweepParams = SweepParams()
    noise         : NoiseParams = NoiseParams()

## 3. The Island Structure

The potential and electrostatic energy of the dot in Beenakker's model:
\begin{equation}
    \phi(Q) = \frac{Q}{C} + \phi_{\text{ext}}
\end{equation}

\begin{equation}
    U(N) = \frac{(Ne)^{2}}{2C} - Ne \phi_{\text{ext}}
\end{equation}

Introducing an externally induced charge $Q_{\text{ext}} = C\phi_{\text{ext}}$:
\begin{equation}
    U(N) = \frac{(Ne - Q_{\text{ext}})^{2}}{2C} - \frac{Q_{\text{ext}}^{2}}{2C}
\end{equation}

Assuming no external potential (yet) and operating in atomic units:
\begin{equation}
    Q = q_{0} + C_{S}V_{S} + C_{D}V_{D} + C_{g}V_{G}
\end{equation}

\begin{equation}
    U(N) = \frac{(N - Q)^{2}}{2C_{\Sigma}}
\end{equation}

### Energy Levels 

**Ground-state energy.** For $N$ electrons on the island,
\begin{equation}
    E(N) = \sum_{p} n_{p}\,\epsilon_{p} + \frac{(N - Q)^{2}}{2C_{\Sigma}}
\end{equation}
- $\epsilon_p$ are the single-particle states and $n_p  = 0,1$ their occupations. The ground state fills the lowest $N$ states.
- The second term is the charging energy. All electron–electron interaction is replaced by one constant capacitance $C_\Sigma$ (Beenakker's constant-interaction model). $Q$ is in units of $e$ and set by the electrode voltages.

We use a uniform ladder whose spacing $\Delta E$ is the mean level spacing of the $\delta$-layer, estimated from its density of states.

**Step 1: subband filling.** The $\delta$-layer is a 2D electron gas with several subbands. The 1D Schrödinger–Poisson solution across the layer (Constantinou et al.) gives the subband bottoms $\epsilon_{n\Gamma}$, $\epsilon_{n\Delta}$. Filling them up to the Fermi energy $\epsilon_F$ (SI Eq. 9):
\begin{align}
    2\,(\epsilon_{F} - \epsilon_{n\Gamma})\,D_{\Gamma} &= N_{n\Gamma} \\
    4\,(\epsilon_{F} - \epsilon_{n\Delta})\,D_{\Delta} &= N_{n\Delta}
\end{align}
$N_{n\Gamma}$, $N_{n\Delta}$ are the electrons per unit area in subband $n$. The factors 2 and 4 are the numbers of $\Gamma$ and $\Delta$ valleys.

**Step 2: density of states per valley** (spin included). A parabolic 2D band has the constant density of states $m/2\pi\hbar^2$ per spin. Doubling for spin, and using $\sqrt{m_\parallel m_\perp}$ for the anisotropic $\Delta$ valleys:
\begin{align}
    D_{\Gamma} &= \frac{m_{\perp}}{\pi\hbar^{2}}, &
    D_{\Delta} &= \frac{\sqrt{m_{\parallel}m_{\perp}}}{\pi\hbar^{2}}
\end{align}
with $m_\parallel = 0.98\,m_e$ and $m_\perp = 0.19\,m_e$ (SI p. S7).

With $\hbar^2/m_e = 0.0762~\text{eV nm}^2$, we get $m_e/\pi\hbar^2 = 4.177~\text{eV}^{-1}\text{nm}^{-2}$. The density of states of one whole subband (all its valleys and both spins) is
\begin{align}
    \mathcal{D}_{\Gamma} &= 2 D_\Gamma = 2 \times 0.19 \times 4.177 = 1.59~\text{eV}^{-1}\text{nm}^{-2} \\
    \mathcal{D}_{\Delta} &= 4 D_\Delta = 4 \times \sqrt{0.98 \times 0.19} \times 4.177 = 7.21~\text{eV}^{-1}\text{nm}^{-2}
\end{align}
With the $1\Gamma$, $2\Gamma$ and $1\Delta$ subbands occupied:
\begin{equation}
    D_{2} = 2\,\mathcal{D}_{\Gamma} + \mathcal{D}_{\Delta} = 3.18 + 7.21 = 10.4~\text{eV}^{-1}\text{nm}^{-2}
\end{equation}
$D_2$ counts every state: both spins and all valleys.

**Step 3: mean level spacing.** An island of area $A = 13 \times 13 = 169~\text{nm}^2$ has $D_2 A$ states per unit energy, so single states are $1/(D_2 A) = 0.57$ meV apart on average. If valley degeneracy is lifted, each orbital level holds two electrons ($g = 2$, spin only), and the levels are
\begin{equation}
    \Delta E = \frac{g}{D_{2}A} = \frac{2}{10.4 \times 169}~\text{eV} = 1.14~\text{meV}
\end{equation}

**Limitations.**
1. $C_\Sigma$ is the same for every electron number and configuration. There is no exchange, and no change of the level structure from interactions. This is Beenakker's assumption.
2. A real 13 nm island has an irregular spectrum (shape, disorder, confinement), so its spacings would fluctuate around the mean.
3. This assumes parabolic bands, a constant density of states over a few spacings, and that the electrical area equals the island area $13 \times 13$ nm$^2$. Edge depletion is ignored.
4. $g = 2$ assumes the valleys are split. The SI model neglects inter-valley coupling, and it says the $1\Gamma$ valley splitting becomes vanishingly small for $\delta$-layers wider than 0.2 nm. The main text quotes DFT values of about 1 meV for non-ideal layers. If valleys are not split, the degeneracy is larger and the addition-energy pattern changes.
5. Temperature = 0 here.
6. $Q$ is linear in the voltages. Screening and defect charges come later.

### Adding one electron

The energy cost to transition from $n$ to $n+1$ electrons:
\begin{equation}
    \mu_{\text{add}}(n) \equiv E(n+1) - E(n) = \epsilon_{\lfloor n/2 \rfloor} + \frac{n + \frac{1}{2} - Q}{C_{\Sigma}}
\end{equation}

Where $\epsilon_{\lfloor n/2 \rfloor}$ represents the level the n+1th electron will go into

Coulomb peak conditions (where it costs no charge for current to flow, so set $\mu_{\text{add}} = 0$):
\begin{align}
    Q_{n} &= n + \frac{1}{2} + C_{\Sigma}\epsilon_{\lfloor n/2 \rfloor} \\
    V_{G}^{(n)} &= \frac{Q_{n} - q_{0} - C_{S}V_{S} - C_{D}V_{D}}{C_{g}}
\end{align}

Addition energy alternating states:
\begin{equation}
    E_{\text{add}}(n) \equiv \mu_{\text{add}}(n) - \mu_{\text{add}}(n-1) = 
    \begin{cases} 
        E_{C} & n \text{ odd} \\ 
        E_{C} + \Delta E & n \text{ even} 
    \end{cases}
\end{equation}

In [5]:
class Island:
    """Capacitances, a ladder of levels, and the energies of the N-electron ground state."""
    def __init__(self, p=Params(), q0=0.0):
        self.p = p
        self.C_sigma = 1.0 / p.E_C                       # E_C = e^2/C_sigma  (e = 1)
        self.C_g = p.alpha_g * self.C_sigma              # alpha_g = C_g/C_sigma
        self.C_S = p.c_S * self.C_sigma                  # source capacitance, c_S is its share of C_sigma
        self.C_D = self.C_sigma - self.C_S - self.C_g    # whatever is left of C_sigma, drain + gate 2 + neighbours
        self.eps = p.dE * np.arange(p.K)                 # level energies  (uniform ladder)
        
        # q0 = background charge on the island, in units of e
        self.q0 = q0

    @classmethod
    def centred(cls, p=Params(), n_ref=6):
        """Island whose diamond with n_ref electrons is centred on V_G = 0 (q0 is an unknown offset in the device).
            This function moves the diamonds by finding what q0 so we can centre the diamonds on V_G = 0"""
        # Build an island with q0 = 0 just to find where its peaks sit
        tmp = cls(p, q0=0.0)
        # The n_ref diamond sits between the peak that fills electron n_ref (index n_ref - 1) and the next one
        # (index n_ref), so its middle is halfway between those two gate voltages
        mid = 0.5 * (tmp.V_peak(n_ref - 1) + tmp.V_peak(n_ref))
        # V_peak = (Q_peak - q0)/C_g, so choosing q0 = C_g * mid shifts that midpoint to V_G = 0
        return cls(p, q0=tmp.C_g * mid)

    def level_of(self, n):
        """Index of the level that takes the (n+1)-th electron when filling from the bottom."""
        # g electrons per level
        return np.asarray(n) // self.p.g

    def Q(self, V_S, V_D, V_G):
        """Charge in units of e:  Q = q0 + C_S V_S + C_D V_D + C_g V_G."""
        return self.q0 + self.C_S * V_S + self.C_D * V_D + self.C_g * V_G

    def E_ground(self, N, Q):
        """Ground-state energy of N electrons: filled ladder + charging energy (N - Q)^2 / 2 C_sigma."""
        N = np.asarray(N)
        g = self.p.g
        # N electrons fill whole levels first (g each) and any left over go into the next level up
        full, rest = N // g, N % g
        # cumulative[k] = energy of the lowest k levels added up (cumulative[0] = 0), so we can read off the filled-levels energy
        cumulative = np.concatenate([[0.0], np.cumsum(self.eps)])
        # Each full level costs g * its energy, then the leftover electrons sit in the next level.
        ladder = g * cumulative[full] + rest * self.eps[np.minimum(full, self.p.K - 1)]
        # Add the electrostatic part of the constant-interaction model
        return ladder + (N - Q) ** 2 / (2.0 * self.C_sigma)

    def mu_add(self, n, Q):
        """Energy to add the (n+1)-th electron:  E(n+1) - E(n) = eps_{n//g} + (n + 1/2 - Q)/C_sigma."""
        # Subtract E(n) from E(n+1): the ladder part leaves just the level the new electron goes into, and the
        # charging part gives (n + 1/2 - Q)/C_sigma
        return self.eps[self.level_of(n)] + (n + 0.5 - Q) / self.C_sigma

    def E_add(self, n):
        """Addition energy mu_add(n) - mu_add(n-1): E_C or E_C + dE """
        # The gap between two neighbouring mu_add values
        return self.mu_add(n, 0.0) - self.mu_add(n - 1, 0.0)

    def Q_peak(self, n):
        """Value of Q at which electron n+1 can enter at zero bias: mu_add(n) = 0."""
        # Set mu_add(n) = 0 and solve for Q
        return n + 0.5 + self.C_sigma * self.eps[self.level_of(n)]

    def V_peak(self, n, V_S=0.0, V_D=0.0):
        """Gate voltage of the Coulomb peak fron n to n+1."""
        # Take Q_peak remove the parts of Q that dont come from the gate and divide by C_g to get volts
        return (self.Q_peak(n) - self.q0 - self.C_S * V_S - self.C_D * V_D) / self.C_g

## 4. Coulomb Diamonds at T=0

To draw the diamonds I sweep the gate voltage and the source bias over a grid and check, at every point, whether the island is stuck at $n$ electrons. It's stuck (Coulomb blockade, no current) when
\begin{equation}
    \mu_{\text{add}}(n) > \max(\mu_{S}, \mu_{D}) \quad \text{and} \quad \mu_{\text{add}}(n-1) < \min(\mu_{S}, \mu_{D})
\end{equation}
The first condition says the next electron can't get in and the second says the last one can't get out. The lead levels are $\mu_S = -V_S$ and $\mu_D = -V_D$ (electrons are negative). `blockade_map` runs this check for every $n$ at every grid point and returns where the island is blocked and which $n$ it holds there.

In [6]:
def blockade_map(isl, V_G, V_S, V_D=0.0):
    """Coulomb-blockade map on the grid V_S (rows) x V_G (columns).
    Returns (blockaded: bool array, n: electron number of the diamond, -1 where current flows).
    an n-electron ground state is blockaded if:
        mu_add(n) > max(mu_S, mu_D)   (cannot add)   and   mu_add(n-1) < min(mu_S, mu_D)   (cannot remove),
    with the source and drain chemical potentials mu_k = -V_k (electrons are negative)."""
    # build the grid and work out Q at every point
    VG, VS = np.meshgrid(V_G, V_S)
    Q = isl.Q(VS, V_D, VG)
    # higher and lower of the two electrode chemical potentials at each point
    mu_hi, mu_lo = np.maximum(-VS, -V_D), np.minimum(-VS, -V_D)
    blockaded = np.zeros(VG.shape, bool); n_map = -np.ones(VG.shape, int)
    # try every electron number and mark the points where it is blockaded
    for n in range(0, isl.p.g * isl.p.K - 1):
        up = isl.mu_add(n, Q)                                              # cost to add the next electron
        down = isl.mu_add(n - 1, Q) if n > 0 else -np.inf * np.ones_like(Q)  # energy released on removing one (n=0 has nothing to remove)
        m = (up > mu_hi) & (down < mu_lo)                                  # cant add and cant remove = stuck at n
        blockaded |= m; n_map[m] = n

    return blockaded, n_map

# 29/10/26

## 5. Finite Temperature

At T=0, the blockade rule means that an electron can only get on it it has at least $mu_\text{add}$ of energy. At a finite temperature T the energy of the electrons are spread around their Fermi Level $E_f$ by the Fermi-Dirac distribution

\begin{equation}
    f(E-E_f) = [1 + \exp{\frac{E-E_f}{kT}}]^{-1}
\end{equation}

This means that for a Coulomb diamond graph, the edges will blur over a few kT and the Coulomb peak graphs will be wider as more electron energies are considered. As kT is much smaller than the charging energy of the island, the position and size of the diamonds do not change.

Now, we give every tunnelling process a rate ($\Gamma$), find the probability P(n) that the electron holds n electrons and we can find the current using this. This is the Beenakker model which also assumes:

- kT and the energy level spacing $\Delta E$ are much larger than $\hbar \Gamma$ so the resonances are thermally broadened and the island is described by what levels are occupied. For $\Delta E = 1.14 meV$ the upper limit is T less than 13K.
- Energy is conserved in every tunnelling event.
- Between tunnelling events the island relaxes to its ground state (strong inelastic scattering on the island, Beenakker Section 5)
- We will have tunnel rates to the source and drain $\Gamma_S$ and $\Gamma_D$ independent of the level of the island.

## The Fermi Function

f(E) = Probability that a state at energy E above the lead's Fermi level is filled, 1-f(E) for it being empty.

We use logarithms to avoid overflow so we get:
\begin{equation}
    \log f(E) = -\log (1 + e^{\frac{E}{kT}})
\end{equation}

\begin{equation}
    \log (1 - f(E)) = -\log (1 + e^{\frac{-E}{kT}})
\end{equation}

In [7]:
from scipy.special import logsumexp

def log_fermi(E,kT):
    """Probability that a lead state at energy E above the lead's Fermi level is filled"""
    return -np.logaddexp(0.0, E/kT)

def log_one_minus_fermi(E,kT):
    """Probability that a lead state at energy E above the lead's Fermi level is not filled"""
    return -np.logaddexp(0.0,-E/kT)

## Counting Spin States

With 2 electrons per level, the number of ways a tunnelling event can happen depends on how full the level is:

- the (n+1)th electron enter level n/2, which has 2 - (n mod 2) empty slots, so it can get on in that many ways
- an island holding electron n has ((n-1) mod 2) + 1 electrons in its top level so it can leave in that many ways

An empty level accepts in 2 ways (spin up or spin down) and a half-filled level accepts in 1 way. A full level can release in 2 ways and a half-filled level can release in one way.

In general level p holds min(max(n-2p,0),2) electrons.

These possibilities will shift each Coulomb peak and lower its height.

In [8]:
def slots_full(n,p,g):
    """Number of electrons in level p when the island holds n electrons in its ground state"""
    return int(np.clip(n - g * p, 0, g))

def slots_empty(n,p,g):
    """Number of empty spin slots in level p when the island holds n electrons in its ground state"""
    return g - slots_full(n,p,g)

## Letting the electron enter higher levels

At T=0 we do not consider excited levels, the electron can only go into the lowest empty level of the island and the last one can only leave from the highest occupied level.

At larger biases the leads supply more energy so electrons can tunnel into a higher, empty level and an electron can leave from a lower, full level. Each extra reachable level is a new way through the island and the current steps up.
These steps come every $Delta E$ of energy so the bias range of +-20 mV opens up to about 17 extra levels.

A map of dI/dV shows that each step is a line parallel to a diamond edge but the sizes and positions of the diamonds do not change as the edges are set by the lowest empty and highest occupied levels.

We also assume the island relaxes to the ground state of n+1 electrons when an electron enters a higher level (as in Beenakker section 5). We use this assumption as the island is a small 13x13nm patch with a small amount of electrons that can relax quickly.

## Rates, Probabilities and Currents
### Rates
Adding an electron to level p of an island holding n electrons is given by $\epsilon_p + (n + 1/2 -Q)/C_{\Sigma}$, an electron leaving level p of an island holding n+1 leaves with the same energy. For p being the lowest energy level this is $\mu_{\text{add}} (n)$ however if p is a higher level it will cost $\Delta E$ more to add the electron onto the island.

For $\mu_S = -V_S$, $\mu_D = -V_D$ and tunnel rates $\Gamma_k$ through barrier k, the total rates between n and n+1 are sums over the levels.
On from lead k:

\begin{equation}
    \sum_p \Gamma_k \times (\text{empty slots in p}) \times f(\epsilon_p + (n + 1/2 -Q)/C_{\Sigma} - \mu_k)
\end{equation}

Off into lead k:

\begin{equation}
    \sum_p \Gamma_k \times (\text{electrons in p}) \times [1 - f(\epsilon_p + (n + 1/2 -Q)/C_{\Sigma} - \mu_k)]
\end{equation}

With the slots counter for n electrons (on) and n+1 electrons (off). Levels further from the top level than the largest bias (plus 30kT) are skipped as the Fermi factors are very close to 0.

### Probabilities

In a steady state the flow from n to n+1 equals the flow back (from Beenakker):

\begin{equation}
    P(n+1) [\text{rate off into S + rate off into D}] = P(n) [\text{rate on from S + rate on from D}]
\end{equation}

Where P(n) is the fraction of time the island holds n electrons.
Only electron numbers the sweep can reach are kept for efficiency

### Current

This is given by the net rate at which electrons tunnel onto the island from the source:

\begin{equation}
    I = -e \sum_n [P(n) \text{rate on from S} - P(n+1) \text{rate off into S}]
\end{equation}

This is positive from source to drain when V_s > 0.
Inside a diamond it is basically = 0 (as expected) and it steps up every time another energy level becomes reachable.

In [9]:
def current_map(isl, V_G, V_S, V_D=0.0, kT=100 * mK, Gamma_S=1.0, Gamma_D=1.0):
    """Current through the island at temperature kT on the grid V_S (rows) x V_G (columns).
    Returns (I, n_mean): the current from source to drain (units of e*Gamma when Gamma_S,
    Gamma_D are in units of Gamma) and the mean number of electrons on the island.
    
    Sequential tunnelling with relaxation (Beenakker Section V): an electron can tunnel on into any
    empty level p, or off from any occupied level p, and the island relaxes to its ground state before the next
    event.
    
    Only the lowest empty / highest occupied level counts at T = 0 and small
    bias, which is blockade_map."""
    # build the grid and work out Q at every point (same as blockade_map)
    VG, VS = np.meshgrid(V_G, V_S)
    Q = isl.Q(VS, V_D, VG)
    mu_S, mu_D = -VS, -V_D  # lead levels (electrons are negative)
    g, K, eps = isl.p.g, isl.p.K, isl.eps

    # electron numbers the sweep can reach: the zero-bias ground state at the two ends of the Q range, +- 4.
    # Further away P(n) is approximately exp(-E_C/kT), which is almost zero so they are skipped.
    n_eq = lambda q: int(np.sum(isl.mu_add(np.arange(g * K - 1), q) < 0))
    n_lo, n_hi = max(n_eq(Q.min()) - 4, 0), min(n_eq(Q.max()) + 4, g * K - 1)
    # levels further from the top level than the largest bias (+30 kT) can never be reached so they are skipped
    reach = np.max(np.abs(mu_S - mu_D)) + 30 * kT

    def log_rates(n):
        """Logs of the total rates between n and n+1 electrons, through each lead, at every grid point."""
        charge = (n + 0.5 - Q) / isl.C_sigma  # charging part of the energy, the same for every level
        top = n // g  # level the next electron enters in the ground state
        on_S = on_D = off_S = off_D = -np.inf  # log(0): no processes added yet
        for p in range(K):
            E = eps[p] + charge  # energy to add an electron to level p (= energy one leaves p with)
            n_in, n_out = slots_empty(n, p, g), slots_full(n + 1, p, g)
            if n_in > 0 and eps[p] - eps[top] <= reach:  # an empty slot in p: an electron can tunnel on into it
                on_S = np.logaddexp(on_S, np.log(Gamma_S * n_in) + log_fermi(E - mu_S, kT))
                on_D = np.logaddexp(on_D, np.log(Gamma_D * n_in) + log_fermi(E - mu_D, kT))
            if n_out > 0 and eps[top] - eps[p] <= reach:  # an electron in p: it can tunnel off from it
                off_S = np.logaddexp(off_S, np.log(Gamma_S * n_out) + log_one_minus_fermi(E - mu_S, kT))
                off_D = np.logaddexp(off_D, np.log(Gamma_D * n_out) + log_one_minus_fermi(E - mu_D, kT))
        return on_S, on_D, off_S, off_D

    # pass 1: P(n) from P(n+1) x (total rate off) = P(n) x (total rate on), as logs (Beenakker Equation 5.1)
    logP, on_S_list, off_S_list = [np.zeros(Q.shape)], [], []
    for n in range(n_lo, n_hi):
        on_S, on_D, off_S, off_D = log_rates(n)
        logP.append(logP[-1] + np.logaddexp(on_S, on_D) - np.logaddexp(off_S, off_D))
        on_S_list.append(on_S)
        off_S_list.append(off_S)  # kept for the current in pass 2
    logP = np.array(logP)  # logP[i] is for n = n_lo + i
    logP -= logsumexp(logP, axis=0)  # normalise so the probabilities add up to 1

    # pass 2: net electrons per unit time onto the island from the source, and the mean electron number
    flow, n_mean = np.zeros(Q.shape), n_lo * np.exp(logP[0])
    for i in range(n_hi - n_lo):
        flow += np.exp(logP[i] + on_S_list[i]) - np.exp(logP[i + 1] + off_S_list[i])
        n_mean += (n_lo + i + 1) * np.exp(logP[i + 1])

    # current = -e x electron flow (e = 1): positive from source to drain when V_S > 0
    return -flow, n_mean


## 6. A stray dopant near island 1 (Hofheinz PhD thesis)

Arsenic devices contain stray As dopants outside the $\delta$-layer. An isolated dopant holds 0 or 1 electron (Hofheinz p. 86), swaps its electron with its nearest lead by slow tunnelling, and acts on island 1 only through its charge. It carries no current. Hofheinz models this as a trap and the device state is now $(n, t)$: $n$ electrons on the island, $t = 0$ or 1 on the dopant.

1. The island feels a fraction $\beta$ of the dopant's electron, so $Q \to Q - \beta$, and every diamond moves up in gate voltage by $\delta V_G = \beta e/C_g$ (Eq. 5.15, p. 71).
2. The gate pulls an electron onto it, and each island electron pushes it off by $\beta E_C$. Over a gate range $\Delta V_G = \beta(1-\beta)E_C/(e\alpha_G)$ (Eq. 5.16, p. 71) it flips as the island fills. Some peaks may disappear, and teeth appear on the diamond edges (pp. 71–73).
3. It switches slowly and at random, so the current jumps between two values: a telegraph signal, like a TLF. Averaged over time, both diamond sets appear wherever it is full only part of the time: the double peaks.

The code calculates: At every $(V_G, V_S)$: the fraction of time the dopant is full, $p$, the current $I = (1-p)\,I_\text{empty} + p\,I_\text{full}$, and the dopant's switching rates (for the noise stage).

**Parameters** (in a new class, `DopantParams`, in the next cell):
- $\beta$, from Coulomb's law at a distance $d$
- the dopant's gate lever arm $\alpha_G$ (Calculated using the same ratio as the trap lever arm to the islands)
- which lead it tunnels to
- the gate voltage $V_c$ around which it fills.

In [10]:
# Parameters for one stray dopant.
# the sweep ranges from SweepParams and uses the temperature T_e from the finite-temperature section.

@dataclass(frozen=True)
class DopantParams:                 # one stray As dopant near island 1 for now
    d       : float = 50 * nm       # dopant-island distance which sets beta by Coulomb's law (beta = 0.22)
    alpha_G : float = 0.004          # gate lever arm on the dopant, C_tG/C_tt 
    lead    : str   = 'D'           # the lead it tunnels to: the grounded drain
    g_t     : int   = 2             # spin: an empty dopant takes either spin, and holds one electron at most
    V_c     : float = 0.0 * mV      # gate voltage at the centre of its switching zone (0 = middle of diamond 6)
    Gamma_t : float = 1e-3          # its tunnel rate to the lead, in units of the island's Gamma (ratio of Hofheinz Eq. 5.14, p. 69)
    a_t     : float = 1.65 * nm     # hydrogenic As radius, only used to give C_tt a size

### 6.1 The energy

**Capacitance network:** The dopant has capacitances $C_{tm}$ to the island and $C_{tG}, C_{tS}, C_{tD}$ to the gate, source and drain. Its total is $C_{tt}$. The electrodes induce a charge on it (in electrons) of
$$Q_t = q_{t0} + C_{tS}V_S + C_{tD}V_D + C_{tG}V_G ,$$
where $q_{t0}$ sets where its level sits. Two fractions matter:
$$\beta = \frac{C_{tm}}{C_{tt}}\, \qquad \alpha_G = \frac{C_{tG}}{C_{tt}}\ $$
Where $\alpha_G$ is the lever arm between the gate and the dopant.

**Energy:** For two conductors the electrostatic energy is $W = \tfrac12 (N - X)^{\mathsf T}\hat C^{-1}(N - X)$, with $N$ the electron numbers and $X$ the induced charges (Eq. 2.7, p. 21). Writing $\hat C = L\,\mathrm{diag}(C_{tt},\,C)\,L^{\mathsf T}$ with $L = \begin{pmatrix}1&0\\-\beta&1\end{pmatrix}$ (Eq. 5.12, p. 68), where $C$ is the island's total capacitance, turns $W$ into two squares (Eq. 5.13, p. 69). With our level ladder added ($e = 1$):
$${E(n,t) = \sum_p n_p\varepsilon_p + \frac{(n + \beta t - Q)^2}{2C_\Sigma} + \frac{(t - Q_t)^2}{2C_{tt}}}$$

We assume:
1. $C_\Sigma = e^2/E_C$ is the island's measured total capacitance, which already includes whatever the dopant adds.
2. $Q$ is our existing island $Q$: the measured lever arms already include the dopant's indirect coupling.

A dopant that holds 0 or 1 electron also has a self-charging term absorbed into $q_{t0}$ (p. 22).

This means that:
- **Dopant frozen in state $t$ (occupied by an electron):** The island at $Q - \beta t$. Every peak moves up in gate voltage by $\delta V_G = \beta e/C_g$ (Eq. 5.15, p. 71).
- **Energy to fill the dopant:** while the island holds $n$ (the energy ladder cancels):
$$\mu_t(n) = E(n,1) - E(n,0) = \beta E_C\Big(n - Q + \frac{\beta}{2}\Big) + \frac{1 - 2Q_t}{2C_{tt}}.$$
The first term is the island's push: each extra electron raises it by $\beta E_C$. The second is the dopant's own level, lowered by $\alpha_G e$ per volt of $V_G$.

### 6.2 Two helpers

- **`log_fstar`** is for the test against Hofheinz's own figure only, whose island is metallic. Between two continua the rate of tunnelling is $\Gamma f^*(\Delta W/kT)$ with $f^*(x) = x/(e^x - 1)$ (Eq. 2.18, p. 24).
- **`n_window`** is lines 25–26 of `current_map`, for several $Q$ arrays at once (we need $Q$ and $Q - \beta$).

In [11]:
import copy
from dataclasses import replace

def log_fstar(x):
    """log of f*(x) = x / (e^x - 1), the rate factor between two METALLIC conductors (Hofheinz Eq. 2.18, p. 24).
    Only used for the test against Hofheinz's metallic island."""
    x = np.asarray(x, float)
    small = np.abs(x) < 1e-6                                       # f*(x) ~ 1 - x/2 near x = 0
    xs = np.where(small, 1.0, x)                                   # placeholder so the logs below never see 0
    with np.errstate(all='ignore'):                               # the branch not chosen may overflow
        big = np.where(xs > 0, np.log(xs) - xs - np.log(-np.expm1(-xs)),   # x > 0:  log x - x - log(1 - e^-x)
                               np.log(-xs) - np.log(-np.expm1(xs)))        # x < 0:  log|x| - log(1 - e^x)
    return np.where(small, -x / 2, big)

def n_window(isl, *Qs):
    """Electron numbers worth keeping for all the induced-charge arrays Qs (as in current_map lines 25-26)."""
    gK = isl.p.g * isl.p.K
    n_eq = lambda q: int(np.sum(isl.mu_add(np.arange(gK - 1), q) < 0))
    n_lo = max(min(n_eq(Q.min()) for Q in Qs) - 4, 0)
    n_hi = min(max(n_eq(Q.max()) for Q in Qs) + 4, gK - 1)
    return n_lo, n_hi

### 6.3 The island with the dopant frozen: `island_chain`

The dopant is about 1000 times slower than the island ($\Gamma_t = 10^{-3}\Gamma$). So while the dopant sits still, empty or full, the island hops on and off thousands of times and settles into its steady state. That means we can first solve the island with the dopant frozen, and only then work out the dopant.

From 6.1, a full dopant is the same as lowering the island's induced charge by $\beta$:

- dopant frozen empty ($t = 0$): the normal island, at induced charge $Q$;
- dopant frozen full ($t = 1$): the normal island, at induced charge $Q - \beta$.

So we need to solve for the island twice, once at $Q$ and once at $Q - \beta$. That is what `island_chain` does.

`current_map` already solves the island, but it can't be used here because:

1. It works out $Q$ itself from the voltages. Here we must input $Q$ *or* $Q - \beta$, so `island_chain` takes $Q$ as an input instead.
2. It does not return $P(n)$, the fraction of time the island holds $n$ electrons; it only returns the current and the mean $n$. The dopant's rates need $P(n)$ (section 6.6), so `island_chain` returns it, as $\log P(n)$.
3. The test against Hofheinz (outputs notebook) needs his metallic island. `metallic=True` swaps in his rates (defined in 6.2) and it is only used for that test.

Apart from these changes, `island_chain` is basically the same as `current_map`

On from lead k:

\begin{equation}
    \sum_p \Gamma_k \times (\text{empty slots in p}) \times f(\epsilon_p + (n + 1/2 -Q)/C_{\Sigma} - \mu_k)
\end{equation}

Off into lead k:

\begin{equation}
    \sum_p \Gamma_k \times (\text{electrons in p}) \times [1 - f(\epsilon_p + (n + 1/2 -Q)/C_{\Sigma} - \mu_k)]
\end{equation}

With the slots counter for n electrons (on) and n+1 electrons (off). Levels further from the top level than the largest bias (plus 30kT) are skipped as the Fermi factors are very close to 0.

### Probabilities

In a steady state the flow from n to n+1 equals the flow back (from Beenakker):

\begin{equation}
    P(n+1) [\text{rate off into S + rate off into D}] = P(n) [\text{rate on from S + rate on from D}]
\end{equation}

Where P(n) is the fraction of time the island holds n electrons.
Only electron numbers the sweep can reach are kept for efficiency

### Current

This is given by the net rate at which electrons tunnel onto the island from the source:

\begin{equation}
    I = -e \sum_n [P(n) \text{rate on from S} - P(n+1) \text{rate off into S}]
\end{equation}

This is positive from source to drain when V_s > 0.

Inputs: the island isl, the induced charge Q ($Q$ or $Q - \beta$) the lead levels mu_S, mu_D, the temperature kT, the tunnel rates Gamma_S, Gamma_D and the range of electron numbers n_lo to n_hi to keep and if the island is metallic or not.

Outputs: logP (logP[i] is $\log P(n_{\rm lo} + i)$) and flow.

The metallic option (test only). Hofheinz's island is metallic: a continuum of states, not a ladder. Then the rate for a hop that costs energy $\Delta E$ is (Hofheinz Eq. 2.18, p. 24)
$$\Gamma_k\, f^*\!\left(\frac{\Delta E}{kT}\right), \qquad f^*(x) = \frac{x}{e^x - 1},$$
with $\Delta E = (n + \frac12 - Q)/C_\Sigma - \mu_k$ for a hop on and $-\Delta E$ for a hop off. This comes from `log_fstar` in 6.2.

In [12]:
def island_chain(isl, Q, mu_S, mu_D, kT, Gamma_S, Gamma_D, n_lo, n_hi, metallic=False):
    """The core of current_map for a given induced charge Q (any array shape) and electron numbers n_lo..n_hi.
    Returns logP (logP[i] is log P(n_lo + i)) and the net number of electrons per unit time onto the island
    from the source. metallic=True uses Hofheinz's metallic-island rates (Eq. 2.18, p. 24), for the test only."""
    g, K, eps = isl.p.g, isl.p.K, isl.eps
    reach = np.max(np.abs(mu_S - mu_D)) + 30 * kT # Maximum occupancy level on the island that can be reached

    def log_rates(n):
        charge = (n + 0.5 - Q) / isl.C_sigma                       # charging part of the energy (as in current_map)
        if metallic:                                               # continuum to continuum: rate = Gamma x f*(energy cost / kT)
            return (np.log(Gamma_S) + log_fstar((charge - mu_S) / kT), np.log(Gamma_D) + log_fstar((charge - mu_D) / kT),
                    np.log(Gamma_S) + log_fstar((mu_S - charge) / kT), np.log(Gamma_D) + log_fstar((mu_D - charge) / kT))
        # from here, identical to current_map section    
        top = n // g  # level the next electron enters in the ground state
        on_S = on_D = off_S = off_D = -np.inf  # log(0): no processes added yet
        for p in range(K):
            E = eps[p] + charge  # energy to add an electron to level p (= energy one leaves p with)
            n_in, n_out = slots_empty(n, p, g), slots_full(n + 1, p, g)
            if n_in > 0 and eps[p] - eps[top] <= reach:  # an empty slot in p: an electron can tunnel on into it
                on_S = np.logaddexp(on_S, np.log(Gamma_S * n_in) + log_fermi(E - mu_S, kT))
                on_D = np.logaddexp(on_D, np.log(Gamma_D * n_in) + log_fermi(E - mu_D, kT))
            if n_out > 0 and eps[top] - eps[p] <= reach:  # an electron in p: it can tunnel off from it
                off_S = np.logaddexp(off_S, np.log(Gamma_S * n_out) + log_one_minus_fermi(E - mu_S, kT))
                off_D = np.logaddexp(off_D, np.log(Gamma_D * n_out) + log_one_minus_fermi(E - mu_D, kT))
        return on_S, on_D, off_S, off_D

    logP, on_S_list, off_S_list = [np.zeros(np.shape(Q))], [], []     # pass 1, as current_map lines 47-54
    for n in range(n_lo, n_hi):                           #P(n) from P(n+1) x (total rate off) = P(n) x (total rate on), as logs (Beenakker Equation 5.1)
        on_S, on_D, off_S, off_D = log_rates(n)
        logP.append(logP[-1] + np.logaddexp(on_S, on_D) - np.logaddexp(off_S, off_D))
        on_S_list.append(on_S); off_S_list.append(off_S)
    logP = np.array(logP)
    logP -= logsumexp(logP, axis=0)
    flow = np.zeros(np.shape(Q))                                   # pass 2, as current_map lines 57-59
    for i in range(n_hi - n_lo):                    # net electrons per unit time onto the island from the source, and the mean electron number
        flow += np.exp(logP[i] + on_S_list[i]) - np.exp(logP[i + 1] + off_S_list[i])
    return logP, flow

### 6.4 The dopant and its capacitances from Coulomb's law

$\beta$ from Coulomb's law. From 6.1, each extra island electron raises the dopant's energy by $\beta E_C$. Seen from a distance $d$ (larger than the island), that electron is a point charge, so the same energy is $U(d) = e^2/(4\pi\varepsilon_0\varepsilon_r d)$ (Coulomb's law between point charges, Hofheinz Eq. 2.12, p. 22). So
$$\beta = \frac{U(d)}{E_C} = \frac{C_\Sigma}{4\pi\varepsilon_0\varepsilon_r d} \approx \frac{11\ \text{nm}}{d}.$$
This is an upper estimate: it assumes uniform silicon and no screening by the leads and gates.

$\beta + \alpha_G + C_{tS}/C_{tt} + C_{tD}/C_{tt} = 1$.
- $\alpha_G$ is chosen so that the ratio between the dopant lever arm and island lever arm is the same as in Hoffheinz's model (ratio = 0.05).
- The rest, $1 - \beta - \alpha_G$, goes to the lead the dopant tunnels to, since it must sit close to that lead.

The code still needs a total, so it uses a sphere of radius $a_t$, $C_{tt} = 4\pi\varepsilon_0\varepsilon_r a_t$, with $a_t$ from `DopantParams` (1.65 nm). It drops out of every result. In atomic units $4\pi\varepsilon_0 = 1$, so $\beta$ is $C_\Sigma / (\epsilon_r d)$.

In [13]:
@dataclass(frozen=True)
class Dopant:
    """One stray dopant that holds 0 or 1 electron, coupled to the island, gate, source and drain by capacitances
    (Hofheinz Fig. 5.3, p. 68), and exchanging electrons with one lead."""
    C_tm    : float                 # dopant - island capacitance
    C_tG    : float                 # dopant - gate capacitance
    C_tS    : float                 # dopant - source capacitance
    C_tD    : float                 # dopant - drain capacitance
    q_t0    : float = 0.5           # background charge induced on the dopant (in e): sets where its level sits
    lead    : str   = 'D'           # the lead it exchanges electrons with ('S' or 'D')
    Gamma_t : float = DopantParams().Gamma_t   # dopant <-> lead tunnel rate, in units of the island's Gamma
    Gamma_tm: float = 0.0           # dopant <-> island tunnel rate (only for the metallic-island test)
    g_t     : int   = 1             # ways into the empty dopant: 1 = non-degenerate (Hofheinz), 2 = spin (As donor)

    @property
    def C_tt(self):                 # total capacitance of the dopant, C_t + C_tm in Hofheinz's notation
        return self.C_tm + self.C_tG + self.C_tS + self.C_tD

    @property
    def beta(self):                 # Hofheinz Eq. 5.11 (p. 68): share of the dopant's capacitance that goes to the island
        return self.C_tm / self.C_tt

    @property
    def alpha_G(self):              # gate lever arm of the dopant (Hofheinz p. 71)
        return self.C_tG / self.C_tt

    def Q_t(self, V_S, V_D, V_G):
        """Charge induced on the dopant by the electrodes, in units of e (Hofheinz's x_t, Eq. 5.10, p. 68)."""
        return self.q_t0 + self.C_tS * V_S + self.C_tD * V_D + self.C_tG * V_G

    @classmethod
    def from_coulomb(cls, isl, d, alpha_G, lead='D', eps_r=ElectrostaticsParams().eps_r_Si, a_t=DopantParams().a_t, **kw):
        """Capacitances from Coulomb's law. beta = U(d)/E_C with U(d) = e^2/(4 pi eps0 eps_r d), the Coulomb energy of
        an island electron and a dopant electron a distance d apart (4 pi eps0 = 1 in atomic units). alpha_G is chosen;
        the rest of the dopant's capacitance goes to its own lead. a_t (the dopant's size) only sets C_tt and drops out."""
        beta = isl.C_sigma / (eps_r * d)                           # = (e^2 / eps_r d) / E_C, since E_C = e^2 / C_sigma
        C_tt = eps_r * a_t                                         # self-capacitance of a sphere of radius a_t
        C_lead = (1.0 - beta - alpha_G) * C_tt                     # what is left goes to the dopant's lead
        assert C_lead > 0, "beta + alpha_G must be below 1"
        return cls(C_tm=beta * C_tt, C_tG=alpha_G * C_tt, C_tS=C_lead if lead == 'S' else 0.0,
                   C_tD=C_lead if lead == 'D' else 0.0, lead=lead, **kw)

### 6.5 Placing the dopant

When the dopant is full (zero bias, island in its lowest state). Two energies compete (Hofheinz p. 71):
- (a) The dopant's own cost: $D(1) - D(0) = (1 - 2Q_t)/2C_{tt}$, which falls steadily by $\alpha_G e$ per volt of $V_G$;
- (b) The island's extra cost: $\Delta M = \min_n E_\text{isl}(n, Q-\beta) - \min_n E_\text{isl}(n, Q)$, which swings up and down once per diamond.

The dopant is full when (a) + (b) < 0.

It flips back and forth while (a) is inside the swing of (b):
$$\Delta V_G = \frac{\Delta M_\text{max} - \Delta M_\text{min}}{e\,\alpha_G}.$$
For a metallic island the swing is $\pm\beta(1-\beta)E_C/2$, so $\Delta V_G = \beta(1-\beta)E_C/(e\alpha_G)$ (Eq. 5.16, p. 71). The ladder makes the swing uneven, so the code finds it numerically.

To centre this range on $V_G = V_c$ (zero bias), set (a) to the middle of the swing:
$$Q_t(V_c) = \tfrac12 + C_{tt}\,\frac{\Delta M_\text{max} + \Delta M_\text{min}}{2}, \qquad q_{t0} = Q_t(V_c) - C_{tG}V_c .$$
The real position of the dopant's level is unknown (binding energies vary from site to site, p. 65), so $V_c$ is a free choice.

**`fixed_dopant_island`** returns the island with the dopant frozen in state $t$ ($q_0 \to q_0 - \beta t$). It is used to draw the empty and full diamond outlines with `blockade_map`.

In [14]:
def dopant_energy(isl, dop, n, t, Q, Q_t):
    """Energy of n island electrons and t (0 or 1) dopant electrons, Hofheinz Eq. 5.13 (p. 69) plus the level ladder:
        E(n, t) = ladder(n) + (n + beta t - Q)^2 / 2 C_sigma + (t - Q_t)^2 / 2 C_tt"""
    return isl.E_ground(n, Q - dop.beta * t) + (t - Q_t) ** 2 / (2.0 * dop.C_tt)

def fixed_dopant_island(isl, dop, t):
    """The island seen with the dopant frozen in state t: an ordinary Island whose background charge is lower by beta t."""
    frozen = copy.copy(isl)
    frozen.q0 = isl.q0 - dop.beta * t
    return frozen

def centre_dopant(isl, dop, V_c):
    """Copy of dop with q_t0 chosen so that its anomaly (Hofheinz Eq. 5.16, p. 71) is centred on V_G = V_c at zero bias.
    Delta M = min_n E_island(n, Q - beta) - min_n E_island(n, Q) oscillates as the island fills and the anomaly runs
    between the gate voltages where D(1) - D(0) = -max(Delta M) and -min(Delta M), so its centre is where
    D(1) - D(0) = -(max + min)/2."""
    n = np.arange(isl.p.g * isl.p.K - 1)
    Q_c = isl.Q(0.0, 0.0, V_c)                                     # island induced charge at the centre
    Qs = Q_c + np.linspace(-1.5, 1.5, 3001) * (1 + isl.C_sigma * isl.p.dE)   # a bit more than one 2-electron period
    dM = np.array([isl.E_ground(n, q - dop.beta).min() - isl.E_ground(n, q).min() for q in Qs])
    dM_mid = 0.5 * (dM.max() + dM.min())
    Q_t_c = 0.5 + dop.C_tt * dM_mid                                # solves (1 - 2 Q_t)/(2 C_tt) = -dM_mid
    return replace(dop, q_t0=Q_t_c - dop.C_tG * V_c)               # Q_t(V_c) = q_t0 + C_tG V_c at zero bias

### 6.6 Rates, the fraction of time full, and the current

Dopant tunnelling to its lead is a single level next to a lead, so it uses the same form as the island's rates (Eqs. 2.19–2.20, pp. 24–25):
$$(n,0)\to(n,1):\ \Gamma_t\,g_t\,f(\mu_t(n) - \mu_L), \qquad (n,1)\to(n,0):\ \Gamma_t\,[1 - f(\mu_t(n) - \mu_L)].$$
Here $\mu_L$ is the lead's Fermi level and $g_t$ the ways in: 2 for an As donor (either spin, one electron), 1 in Hofheinz's test (p. 69).
- The dopant is slow, $\Gamma_t = 10^{-3}\Gamma$ (as Eq. 5.14, p. 69; from `DopantParams`).
- There are no dopant to island hops. For our island that would be level ↔ level, a delta function (Eq. 2.21, p. 25), so it is left out. It is included only in the metallic test, as Hofheinz did.

The dopant stays empty or full for long stretches, and the island settles within each one.
1. The island is original island at $Q$ or $Q - \beta$. This gives $P(n\,|\,t)$ and the source flow (`island_chain`).
2. The dopant's average rates:
$$K_\text{in} = \sum_n P(n|0)\,\Gamma_t g_t f(\mu_t(n) - \mu_L), \qquad K_\text{out} = \sum_n P(n|1)\,\Gamma_t[1 - f(\mu_t(n) - \mu_L)].$$
3. Fraction of time full and current:
$$p = \frac{K_\text{in}}{K_\text{in} + K_\text{out}} = \frac{\tau_\text{full}}{\tau_\text{full} + \tau_\text{empty}}\ \text{(p. 78)}, \qquad I = (1-p)\,I_\text{empty} + p\,I_\text{full}.$$

This is Hofheinz's equation $\Gamma P = 0$ (Eqs. 2.22–2.27, pp. 25–26) in the slow-dopant limit. Against the full solution the error is $\sim\Gamma_t/\Gamma \approx 10^{-5}$. With a single lead, $\Gamma_t$ cancels in $p$ and $I$: it only sets the switching times $1/K_\text{in}$ and $1/K_\text{out}$, which are returned for the noise stage. The dopant itself adds no net current.

In [15]:
def dopant_map(isl, dop, V_G, V_S, V_D=0.0, kT=100 * mK, Gamma_S=1.0, Gamma_D=1.0, metallic=False):
    """Current through the island with one slow stray dopant, on the grid V_S (rows) x V_G (columns).
    Returns (I, occ, n_mean, k_in, k_out): the current (units of e*Gamma), the fraction of time the dopant is full,
    the mean island electron number, and the dopant's rates of filling and emptying (its TLF rates, units of Gamma)."""
    VG, VS = np.meshgrid(V_G, V_S)
    Q, Q_t = isl.Q(VS, V_D, VG), dop.Q_t(VS, V_D, VG)                 # induced charges on the island and the dopant
    mu_S, mu_D = -VS, -V_D                                             # lead levels (electrons are negative)
    mu_L = mu_S if dop.lead == 'S' else mu_D                          # Fermi level of the dopant's own lead
    b = dop.beta
    n_lo, n_hi = n_window(isl, Q, Q - b)                               # electron numbers needed for either dopant state
    ns = np.arange(n_lo, n_hi + 1)

    # step 1: the island with the dopant frozen empty (t = 0) and frozen full (t = 1), as in current_map
    logP0, flow0 = island_chain(isl, Q, mu_S, mu_D, kT, Gamma_S, Gamma_D, n_lo, n_hi, metallic)
    logP1, flow1 = island_chain(isl, Q - b, mu_S, mu_D, kT, Gamma_S, Gamma_D, n_lo, n_hi, metallic)

    # step 2: the dopant's rates of filling and emptying, averaged over the island's P(n | t)
    E = lambda n, t: dopant_energy(isl, dop, n, t, Q, Q_t)
    log_in = log_out = -np.inf                                         # log(0): nothing added yet
    for i, n in enumerate(ns):
        mu_t = E(n, 1) - E(n, 0)                                       # energy to put an electron on the dopant
        log_in = np.logaddexp(log_in, logP0[i] + np.log(dop.Gamma_t * dop.g_t) + log_fermi(mu_t - mu_L, kT))
        log_out = np.logaddexp(log_out, logP1[i] + np.log(dop.Gamma_t) + log_one_minus_fermi(mu_t - mu_L, kT))
        if dop.Gamma_tm > 0 and i > 0:                                 # island n -> n-1 while dopant 0 -> 1 (test only)
            log_in = np.logaddexp(log_in, logP0[i] + np.log(dop.Gamma_tm * dop.g_t) + log_fermi(E(n - 1, 1) - E(n, 0), kT))
        if dop.Gamma_tm > 0 and n < n_hi:                              # dopant 1 -> 0 while island n -> n+1 (test only)
            log_out = np.logaddexp(log_out, logP1[i] + np.log(dop.Gamma_tm) + log_fermi(E(n + 1, 0) - E(n, 1), kT))

    # step 3: fraction of time full, then mix the two island results
    occ = np.exp(log_in - np.logaddexp(log_in, log_out))
    I = -((1 - occ) * flow0 + occ * flow1)                             # current = -e x electron flow from the source
    n_t = [np.sum(ns[:, None, None] * np.exp(lp), axis=0) for lp in (logP0, logP1)]
    n_mean = (1 - occ) * n_t[0] + occ * n_t[1]
    return I, occ, n_mean, np.exp(log_in), np.exp(log_out)

# 12/11/26

## 7. Many stray dopants

Here there are $N$ of them. $N$ is a parameter because the number in island 1 is not known yet.

- The state is which dopants are full: $T = (t_0, \dots, t_{N-1})$ with $t_i = 0$ (empty) or $1$ (full), so $M = 2^N$ states.
- Each dopant is an ordinary section 6 `Dopant`, with its own position, lead (the one on its side), $\beta_i$ from Coulomb's law, level and tunnel rate $\Gamma_{t,i}$. The rates are spaced evenly in log: $\Gamma_{t,i} = \Gamma_\text{max}\,(\Gamma_\text{min}/\Gamma_\text{max})^{i/(N-1)}$.
- They push on each other, through the island ($\beta_i\beta_jE_C$) and by direct Coulomb repulsion $U_{ij} = e^2/(4\pi\varepsilon_0\varepsilon_r r_{ij})$.
- They can hop between each other (Miller–Abrahams rates).
- The kinetic map: the rates $W(T \to T')$ between all $2^N$ states, each averaged over the island as in section 6. Its steady state $P(T)$ gives the current $I = \sum_T P(T)\,I(T)$.

- Each dopant's level can be tuned so that its rate in equals its rate out at one point, for the most noise (7.5). This works when the dopants are independent, but not when they interact (7.6).

All dopants are slow ($\Gamma_{t,i} \ll \Gamma$), so the island settles in every dopant state first.

In [16]:
# Parameters for many stray dopants
@dataclass(frozen=True)
class ManyDopantParams:
    N         : int   = 5             # number of stray dopants (not known yet: change freely, cost grows as 2^N)
    r_min     : float = 40 * nm       # dopants are placed at random between r_min and r_max from the island centre
    r_max     : float = 100 * nm
    alpha_G   : float = 0.004         # gate share of every dopant, as for one dopant
    g_t       : int   = 2
    Gamma_max : float = 1e-2          # tunnel rates spaced evenly in log from Gamma_max down to Gamma_min
    Gamma_min : float = 1e-6          #   units of Gamma
    V_c_max   : float = 450 * mV      # for the maps, each dopant's level is centred at a random V_G within +- V_c_max
    U_scale   : float = 1.0           # multiplies the bare Coulomb energy U_ij between dopants (0 switches it off)
    nu_hop    : float = 1.0           # Miller-Abrahams prefactor for hopping between dopants, units of Gamma
    seed      : int   = 3             # the random positions and levels

### 7.1 Placing the dopants: `DopantSet`, `inside_lead`, `place_dopants`

`DopantSet` holds the $N$ dopants (each a section 6 `Dopant`), their positions, the Coulomb energy $U_{ij}$ of every pair and the hopping prefactor of every pair.

`place_dopants`, all from one seed so the same seed gives the same dopants:
1. Positions at random between `r_min` and `r_max` from the island centre, evenly per unit area, never inside the source or drain strips (`inside_lead`).
2. Lead: the source if $x < 0$, the drain if $x > 0$.
3. Capacitances from Coulomb's law at the distance $r_i$, as `Dopant.from_coulomb`: $\beta_i = C_\Sigma/(4\pi\varepsilon_0\varepsilon_r r_i)$, gate share $\alpha_G$, the rest to its lead.
4. Tunnel rates spaced evenly in log from `Gamma_max` to `Gamma_min`. A rate goes as $e^{-2\kappa r}$, so dopants spread evenly in distance have rates spread evenly in log, and that spread is what gives $1/f$ noise.
5. Levels not known: each is centred (`centre_dopant`) at a random gate voltage in $\pm$ `V_c_max`, so that its anomaly lands somewhere in the map.
6. Coulomb energy of every pair, $U_{ij} = e^2/(4\pi\varepsilon_0\varepsilon_r r_{ij})$, times `U_scale`.
7. Hopping prefactor of every pair, $\nu\,e^{-2r_{ij}/a}$, with $a = \hbar/\sqrt{2m^*E_b} = 1.93$ nm the size of the bound state.

In [17]:
@dataclass(frozen=True)
class DopantSet:
    """N stray dopants: each an ordinary Dopant (section 6), plus their positions and what couples them."""
    dops : tuple                      # the N Dopant objects
    xy   : np.ndarray                 # positions in the plane, shape (N, 2)
    U    : np.ndarray                 # Coulomb energy between dopant i and dopant j when both are full, shape (N, N)
    H    : np.ndarray                 # Miller-Abrahams hopping prefactor nu e^{-2 r_ij / a}, shape (N, N), units of Gamma
    V_c  : np.ndarray = None          # where each dopant's level was centred (for the maps)

    @property
    def N(self): # list of all dopants
        return len(self.dops)

    @property
    def betas(self): # List of all beta values
        return np.array([d.beta for d in self.dops])

def inside_lead(x, y, geo=GeometryParams()):
    """True if (x, y) lies inside the source or drain strip (a dopant there would be part of the lead)."""
    x_S = -(geo.island_size / 2 + geo.gap_S_rows * geo.row_pitch) # where the source strip starts
    x_D = geo.island_size / 2 + geo.gap_D_rows * geo.row_pitch # where the drain strip starts
    return abs(y) <= geo.strip_width / 2 and (x <= x_S or x >= x_D) 

def place_dopants(isl, mp, hyd, geo=GeometryParams(), eps_r=ElectrostaticsParams().eps_r_Si, a_t=DopantParams().a_t):
    """Put mp.N dopants at random (seed mp.seed) between mp.r_min and mp.r_max from the island centre, outside the leads.
    Each uses the lead on its side (x < 0: source, x > 0: drain), and its level is centred at a random V_G."""
    rng = np.random.default_rng(mp.seed)
    xy = []
    while len(xy) < mp.N: # draws positions until there are N dopants outside the leads
        r = np.sqrt(rng.uniform(mp.r_min ** 2, mp.r_max ** 2))   # uniform per unit area
        th = rng.uniform(0, 2 * np.pi)
        x, y = r * np.cos(th), r * np.sin(th)
        if not inside_lead(x, y, geo): # if the dopant is not inside the leads it is accepted
            xy.append((x, y))
    xy = np.array(xy)
    Gammas = np.logspace(np.log10(mp.Gamma_max), np.log10(mp.Gamma_min), mp.N)   # evenly spaced in log
    V_cs = rng.uniform(-mp.V_c_max, mp.V_c_max, mp.N) # randon centre for each level
    dops = []
    for (x, y), G_t, V_c in zip(xy, Gammas, V_cs): # initialises the dopant and if it is assigned to the lead or drain
        d = Dopant.from_coulomb(isl, d=np.hypot(x, y), alpha_G=mp.alpha_G, lead='S' if x < 0 else 'D',
                                eps_r=eps_r, a_t=a_t, g_t=mp.g_t, Gamma_t=G_t)
        dops.append(centre_dopant(isl, d, V_c)) # centres the dopant
    r_ij = np.hypot(xy[:, None, 0] - xy[None, :, 0], xy[:, None, 1] - xy[None, :, 1]) # radial distance from island
    off = ~np.eye(mp.N, dtype=bool)
    U = np.where(off, mp.U_scale / (eps_r * np.where(off, r_ij, 1.0)), 0.0)   # e^2 / (4 pi eps0 eps_r r_ij), atomic units
    a = 1 / np.sqrt(2 * hyd.m_eff * hyd.E_b)  # bound-state decay length, 1.93 nm. Using effective mass and hydrogenic binding energy
    H = np.where(off, mp.nu_hop * np.exp(-2 * r_ij / a), 0.0) # Miller-Abrahams hopping prefactor
    return DopantSet(tuple(dops), xy, U, H, V_cs)

### 7.2 States, energies and the kinetic map: `configs`, `config_energy`, `chain_log_rates`

States (`configs`). A state $T = (t_1, \dots, t_N)$ says which dopants are full ($t_i = 1$) and which are empty ($t_i = 0$). $T + i$ means $T$ with dopant $i$ filled, $T - i$ with it emptied.

Energy (`config_energy`) with $n$ island electrons: every full dopant lowers the island's charge by its $\beta_i$, every dopant has its own charging term ($Q_{ti}$ is its induced charge and $C_{tt,i}$ its total capacitance, as in section 6), and every full pair repels by $U_{ij}$:
$$E(n, T) = E_\text{isl}\Big(n, Q - \sum_i \beta_i t_i\Big) + \sum_i \frac{(t_i - Q_{ti})^2}{2C_{tt,i}} + \sum_{i<j} U_{ij}\,t_i t_j .$$
So when dopant $j$ fills, dopant $i$'s level rises by $\beta_i\beta_jE_C$ (through the island) plus $U_{ij}$ (directly).

Moves (`chain_log_rates`). Each is averaged over the island, as in section 6.6: $P(n|T)$ is the chance the island holds $n$ electrons while the dopants are in state $T$. $\Delta E_n$ is the energy the move costs with the island held at $n$. $F(\epsilon) = 1/(e^{\epsilon/kT} + 1)$ is the Fermi function (called $f$ in section 6, renamed here because $f$ is now frequency), and $\mu_{L,i}$ is the Fermi level of dopant $i$'s lead.
- fill dopant $i$ from its lead: $\sum_n P(n|T)\,\Gamma_{t,i}\,g_t\,F(\Delta E_n - \mu_{L,i})$,
- empty it into its lead: $\sum_n P(n|T)\,\Gamma_{t,i}\,[1 - F(-\Delta E_n - \mu_{L,i})]$,
- hop an electron from full $i$ to empty $j$ (Miller–Abrahams): $\sum_n P(n|T)\,\nu e^{-2r_{ij}/a}\min(1, e^{-\Delta E_n/kT})$.

Everything is in logs, since rates run from $10^{-2}$ to below $10^{-300}$. For one dopant this is exactly step 2 of `dopant_map`.

In [18]:
def configs(N):
    """All 2^N dopant states as rows of 0/1, row k = the binary digits of k (dopant i is digit i)."""
    k = np.arange(2 ** N) # all state numbers
    return (k[:, None] >> np.arange(N)) & 1

def config_energy(isl, ds, n, b, Q, Q_t):
    """E(n, T) for island number n and dopant state b (a row of configs): the one-dopant energy of section 6
    for every dopant, plus the Coulomb energy U_ij of every pair that is full."""
    E = isl.E_ground(n, Q - np.dot(ds.betas, b)) # island, shifted by every full dopant
    for i, d in enumerate(ds.dops):
        E = E + (b[i] - Q_t[i]) ** 2 / (2.0 * d.C_tt) # each dopant's own charging energy
    return E + 0.5 * b @ ds.U @ b  # each full pair once: sum_{i<j} U_ij b_i b_j

def chain_log_rates(isl, ds, Q, Q_t, mu_S, mu_D, kT, Gamma_S=1.0, Gamma_D=1.0):
    """The kinetic map: log of the rate from every dopant state T to every other T' (shape (M, M, ...) with
    M = 2^N, -inf where there is no direct move), plus the island's source flow in every T (shape (M, ...)).
    Each rate is averaged over the island's P(n | T), as for one dopant (section 6.6)."""
    B = configs(ds.N) # all the states, one row each
    M = len(B)
    n_lo, n_hi = n_window(isl, Q, Q - ds.betas.sum())  # covers every dopant state
    ns = np.arange(n_lo, n_hi + 1) # range of electron numbers
    shape = np.shape(Q) # grid of Q
    logW = np.full((M, M) + shape, -np.inf) # log-rates
    flow = np.zeros((M,) + shape) # flow of electrons from source
    for k, b in enumerate(B):
        # Calculate the log rates for the time the state (n|T) is occupied
        logP, flow[k] = island_chain(isl, Q - np.dot(ds.betas, b), mu_S, mu_D, kT, Gamma_S, Gamma_D, n_lo, n_hi)
        E_k = [config_energy(isl, ds, n, b, Q, Q_t) for n in ns]
        for i, d in enumerate(ds.dops):
            mu_L = mu_S if d.lead == 'S' else mu_D # the source or drain Fermi level
            b2 = b.copy()
            b2[i] = 1 - b[i] # filled if empty, empty if full
            k2 = int(b2 @ (1 << np.arange(ds.N))) # number of the new flipped state
            acc = -np.inf
            for m, n in enumerate(ns):
                dE = config_energy(isl, ds, n, b2, Q, Q_t) - E_k[m]  # energy of the move with n held
                if b[i] == 0:  # fill dopant i from its lead
                    lr = np.log(d.Gamma_t * d.g_t) + log_fermi(dE - mu_L, kT)
                else:    # empty dopant i into its lead
                    lr = np.log(d.Gamma_t) + log_one_minus_fermi(-dE - mu_L, kT)
                acc = np.logaddexp(acc, logP[m] + lr)
            logW[k, k2] = acc # storing the rate
            for j in range(ds.N):   # hop: electron from full i to empty j
                if b[i] == 1 and b[j] == 0 and ds.H[i, j] > 0: # i full, j empty
                    b3 = b.copy()
                    b3[i], b3[j] = 0, 1
                    k3 = int(b3 @ (1 << np.arange(ds.N)))
                    acc = -np.inf
                    for m, n in enumerate(ns):
                        dE = config_energy(isl, ds, n, b3, Q, Q_t) - E_k[m] # change in energy after hop
                        acc = np.logaddexp(acc, logP[m] + np.log(ds.H[i, j]) - np.maximum(dE, 0) / kT) # P(n|T) using Miller-Abrahams rate
                    logW[k, k3] = acc
    return logW, flow # returns log-rates and flows

### 7.3 The steady state: `stationary`

What it finds. $P(T)$, the fraction of the time the dopants spend in each state $T$ (called `P_T` in the code). In the long run the moves into each state balance the moves out. For one dopant that gave $p = K_\text{in}/(K_\text{in} + K_\text{out})$. For $2^N$ states it is $2^N$ equations for the $P(T)$, with $\sum_T P(T) = 1$.

The rates differ by hundreds of orders of magnitude, and an ordinary linear solver subtracts numbers of very different sizes and loses all accuracy.

The GTH method (Grassmann, Taksar and Heyman 1985). Number the states $0, 1, \dots, M-1$ (with $M = 2^N$) and write $q_{ij}$ for the rate from state $i$ to state $j$.
1. Remove the states one at a time, from the last down to state 1. Removing state $k$ redirects every path that went through it: the rate from $i$ to $j$ gains (rate $i \to k$) $\times$ (chance that from $k$ it goes on to $j$),
$$q_{ij} \to q_{ij} + \frac{q_{ik}\,q_{kj}}{R_k}, \qquad R_k = \sum_{j<k} q_{kj},$$
where $R_k$ is the total rate out of $k$ to the states still left.
2. Build the answer back up: $P_0 = 1$, then $P_k = \sum_{i<k} P_i\,q_{ik}/R_k$ for $k = 1, 2, \dots$
3. Divide by $\sum_k P_k$ so they add up to 1.

It only adds, multiplies and divides positive numbers, so nothing is lost. It works on every grid point at once.

In [19]:
def stationary(logW, floor=-700.0):
    """P(T), the fraction of time spent in every dopant state T: the steady state of the kinetic map, by the GTH method
    (Grassmann, Taksar and Heyman 1985), which never subtracts and so stays accurate when rates differ by hundreds
    of orders of magnitude. Works on every grid point at once."""
    M = logW.shape[0] # number of states
    q = np.where(np.isfinite(logW), np.exp(np.maximum(logW, floor)), 0.0)   # rates, floor keeps every move > 0
    q = np.moveaxis(q, (0, 1), (-2, -1)).copy()  # shape (..., M, M)
    R = np.zeros(q.shape[:-1]) # R_k array
    for k in range(M - 1, 0, -1): # remove states M-1, ..., 1 in turn
        R[..., k] = q[..., k, :k].sum(axis=-1)
        q[..., :k, :k] += q[..., :k, k, None] * q[..., k, None, :k] / R[..., k, None, None]
    P_T = np.zeros(q.shape[:-1])
    P_T[..., 0] = 1.0
    for k in range(1, M): # build the weights back up
        P_T[..., k] = np.einsum('...i,...i->...', P_T[..., :k], q[..., :k, k]) / R[..., k]
        P_T[..., :k + 1] /= P_T[..., :k + 1].max(axis=-1, keepdims=True)   # rescale as we go, so nothing overflows
    P_T /= P_T.sum(axis=-1, keepdims=True)
    return np.moveaxis(P_T, -1, 0)  # shape (M, ...)

### 7.4 The map: `many_dopant_map`

The same idea as `dopant_map`, for $N$ dopants, on the grid $V_S \times V_G$, a few rows at a time to save memory:
1. the kinetic map at every point (`chain_log_rates`),
2. its steady state $P(T)$ (`stationary`),
3. the current, each state's island current weighted by the time spent there, $I = \sum_T P(T)\,I(T)$,
4. each dopant's fraction of time full, $p_i = \sum_T P(T)\,t_i$.

Returns the current, the mean number of full dopants and the $p_i$. The work doubles with every extra dopant: about 3 minutes for 5 dopants on a 1 mV by 0.25 mV grid.

`interacting=False` makes the dopants independent (`independent_rates`): every dopant fills and empties at the rates it has with all the others empty, whatever the others are doing, and there are no hops. So no dopant pushes on another, through the island or directly. A full dopant still shifts the island, so the current still feels every dopant. This is used for the map with every dopant tuned (7.5).

In [20]:
def independent_rates(logW, N):
    """The rate table with the dopants made independent: every dopant fills and empties at its own rates from the
    all-empty state, whatever the others are doing, and there are no hops. logW[k, k2, ...] as from chain_log_rates."""
    out = np.full_like(logW, -np.inf) # no moves to start with
    ones = 1 << np.arange(N) # 2^i: the state with only dopant i full
    for k in range(2 ** N):
        for i in range(N):
            if (k >> i) & 1 == 0: # dopant i empty in state k: it fills at its rate from the all-empty state
                out[k, k + ones[i]] = logW[0, ones[i]]
            else: # dopant i full in state k: it empties at its rate back to the all-empty state
                out[k, k - ones[i]] = logW[ones[i], 0]
    return out

def many_dopant_map(isl, ds, V_G, V_S, V_D=0.0, kT=100 * mK, Gamma_S=1.0, Gamma_D=1.0, rows=8, interacting=True):
    """Current with N slow dopants on the grid V_S (rows) x V_G (columns), a few rows at a time to save memory.
    Returns (I, n_full, p): current (e Gamma), mean number of full dopants, and each dopant's fraction of time full.
    interacting=False: the dopants made independent (independent_rates)."""
    B = configs(ds.N) # all states
    I = np.zeros((len(V_S), len(V_G))) # empty current map, one row for V_s and one column for V_G
    p = np.zeros((ds.N,) + I.shape) # empty map of dopants fraction of time full
    for r0 in range(0, len(V_S), rows):
        VG, VS = np.meshgrid(V_G, V_S[r0:r0 + rows])
        Q = isl.Q(VS, V_D, VG) # island induced charge
        Q_t = [d.Q_t(VS, V_D, VG) for d in ds.dops] # dopants induced charge
        logW, flow = chain_log_rates(isl, ds, Q, Q_t, -VS, -V_D, kT, Gamma_S, Gamma_D) # rates and flows
        if not interacting:
            logW = independent_rates(logW, ds.N) # no pushing between dopants, no hops
        P_T = stationary(logW) # fraction of time in each dopant state
        I[r0:r0 + rows] = -(P_T * flow).sum(axis=0) # current = -e x mean electron flow
        p[:, r0:r0 + rows] = np.tensordot(B.T, P_T, axes=1) # p_i = sum over states with dopant i full
    return I, p.sum(axis=0), p

### 7.5 Tuning each dopant for the most noise: `OperatingParams`, `flank_point`, `balance_dopant`, `balance_each`

A dopant makes the most noise when it is full half the time, $p = 1/2$, because the noise is $\delta V_G^2\,p(1-p)$. 

Since $p = K_\text{in}/(K_\text{in} + K_\text{out})$, $p = 1/2$ is the same as rate in = rate out.

How full a dopant is depends on its level compared with its lead's Fermi level:

- level well below the Fermi level: always full,
- level well above: always empty,
- level within a few $kT$ of it ($kT = 8.6\ \mu$eV): switching, and full half the time when they line up.

Tuning slides the level, through the dopant's background charge $q_{t0}$, until $p = 1/2$ (`balance_dopant`, using Brent's method). The tunnel rates are not changed.

It only holds at the point we choose, because the gate also moves the level, by about $2.5\,kT$ per mV. A dopant tuned to $p = 1/2$ is already at $p = 0.11$ 1 mV below that gate voltage and $p = 0.85$ 1 mV above. So the tuning is done where a noise measurement would sit (`OperatingParams`, `flank_point`): $V_S = 0.1$ mV, at the steepest point of the 6 to 7 peak. There, $K_\text{in} = K_\text{out} = 0.571\,\Gamma_t$.

`balance_each` tunes each dopant on its own, with the others empty. That is only right if the dopants do not push on each other (7.6).

In [21]:
# Parameters for tuning the dopants
from scipy.optimize import brentq

@dataclass(frozen=True)
class OperatingParams:                # the point the dopants are tuned at (where a noise measurement would sit)
    V_S_op : float = 0.1 * mV         # small source bias, so there is a current peak to sit on
    n_peak : int   = 6                # the Coulomb peak between diamonds n_peak and n_peak + 1

def flank_point(V_G, I):
    """The steepest rising point of a current peak along V_G. Returns its index and the slope dI/dV_G there."""
    slope = np.gradient(I, V_G)
    k = int(np.argmax(slope))
    return k, slope[k]

def balance_dopant(isl, dop, V_G, V_S, V_D=0.0, kT=100 * mK):
    """Copy of dop with q_t0 chosen so that K_in = K_out, i.e. p = 1/2, at the one point (V_G, V_S) to get the most
    noise. Raising q_t0 lowers the dopant's level, so p rises steadily from 0 to 1 with it."""
    def excess(q):  # p - 1/2 for a trial q_t0
        _, occ, _, _, _ = dopant_map(isl, replace(dop, q_t0=q), np.array([V_G]), np.array([V_S]), V_D, kT)
        return occ.item() - 0.5 # p - 1/2
    # solve for p-1/2 = 0 to get equal occupation and non-occupation
    q = brentq(excess, dop.q_t0 - 1.0, dop.q_t0 + 1.0, xtol=1e-12)    # the q_t0 where p = 1/2
    return replace(dop, q_t0=q)

def balance_each(isl, ds, V_G, V_S, V_D=0.0, kT=100 * mK):
    """Copy of the set with every dopant's level chosen so that, on its own, it is full half of the time at (V_G, V_S):
    the most noise from each. Uses balance_dopant (one dopant) for each in turn."""
    return replace(ds, dops=tuple(balance_dopant(isl, d, V_G, V_S, V_D, kT) for d in ds.dops))

### 7.6 Tuning every dopant at once with the interactions on: `point_chain`, `stationary_point`, `balance_all`

#### A dopant's rates depend on the others

With the interactions on, when dopant $j$ fills, dopant $i$'s level rises by $\beta_i\beta_jE_C + U_{ij}$, which is 100 to 600 times $kT$ here (7.2). So dopant $i$ has one fill rate and one empty rate while $j$ is empty, and completely different ones while $j$ is full. Rate in = rate out then has to mean the average rates, over the states where dopant $i$ is empty (in) and full (out), weighted by the time spent in each:

$$\bar K_{\text{in},i} = \frac{\sum_{T:\,t_i = 0} P(T)\,W(T \to T+i)}{\sum_{T:\,t_i = 0} P(T)}, \qquad \bar K_{\text{out},i} = \frac{\sum_{T:\,t_i = 1} P(T)\,W(T \to T-i)}{\sum_{T:\,t_i = 1} P(T)}.$$

In the steady state dopant $i$ fills as often as it empties, $(1-p_i)\,\bar K_{\text{in},i} = p_i\,\bar K_{\text{out},i}$, so this is again the same as $p_i = 1/2$.

#### Why they cannot all be at $p = 1/2$

A rise of 100 to 600 $kT$ puts dopant $i$'s level far above its lead's Fermi level, so while $j$ is full, $i$ cannot fill. The dopants take turns. Tuned one by one (7.5) and then with the interactions on, the five share six states (all empty, or one of the five full), $1/6$ of the time each. So each $p_i = 1/6$, not $1/2$.

#### The rates at one point (`point_chain`, `stationary_point`)

`point_chain` is the kinetic map of 7.2 at the one point, as a $2^N \times 2^N$ table of rates $W(T \to T')$. `stationary_point` is its steady state $P(T)$, by the GTH method of 7.3.

#### The attempt (`balance_all`)

Change each dopant's level in turn, with Brent's method as in 7.5, until its $p_i = 1/2$ with the interactions on, and repeat for up to 10 rounds. Tuning one dopant moves the others away from $1/2$ again.

#### What happens

It cannot get there. It ends with exactly one dopant full at all times, so the $p_i$ add up to 1, while five dopants at $1/2$ would need 2.5. Swapping which one is full is extremely slow, so the average rates collapse to $10^{-17}$ to $10^{-21}\,\Gamma$ and the dopants hardly switch.

So every dopant can be tuned to rate in = rate out only if they do not interact.

In [22]:
def point_chain(isl, ds, V_G, V_S, V_D=0.0, kT=100 * mK, interacting=True):
    """The kinetic map at one point (V_G, V_S): the rate matrix W (W[k, k2] = rate from state k to k2, units of Gamma)
    and the island current in every dopant state. interacting=False: the dopants made independent (independent_rates)."""
    Q = isl.Q(V_S, V_D, V_G) # island induced charge
    Q_t = [d.Q_t(V_S, V_D, V_G) for d in ds.dops] # dopant induced charge
    logW, flow = chain_log_rates(isl, ds, Q, Q_t, -V_S, -V_D, kT) # rates and flows
    if not interacting:
        logW = independent_rates(logW, ds.N) # no pushing between dopants, no hops
    W = np.where(np.isfinite(logW), np.exp(np.maximum(logW, -700.0)), 0.0) # floor 
    return W, -flow

def stationary_point(W):
    """Steady state of one rate matrix W (GTH, as stationary)."""
    with np.errstate(divide='ignore'):
        return stationary(np.log(W)[..., None])[:, 0]

def balance_all(isl, ds, V_G, V_S, V_D=0.0, kT=100 * mK, sweeps=10, tol=1e-6):
    """Copy of the set with every dopant's level chosen so that, with the interactions on, each dopant fills as
    often as it empties at (V_G, V_S): its average K_in equals its average K_out, so p_i = 1/2 for every i.
    One dopant at a time (keeping the others fixed), repeated until no p_i moves by more than tol."""
    B = configs(ds.N) # all states
    def p_all(d_set):
        W, _ = point_chain(isl, d_set, V_G, V_S, V_D, kT, interacting=True)
        return B.T @ stationary_point(W) # returns p_i for a dopant set
    
    # this function tries to set p=1/2 for every dopant
    for sweep in range(sweeps):
        for i in range(ds.N):
            def excess(q):
                trial = replace(ds, dops=tuple(replace(d, q_t0=q) if k == i else d for k, d in enumerate(ds.dops)))
                return p_all(trial)[i] - 0.5
            q = brentq(excess, ds.dops[i].q_t0 - 1.0, ds.dops[i].q_t0 + 1.0, xtol=1e-10)
            ds = replace(ds, dops=tuple(replace(d, q_t0=q) if k == i else d for k, d in enumerate(ds.dops)))
        if np.max(np.abs(p_all(ds) - 0.5)) < tol:
            break
    return ds, sweep + 1